In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '2_train_clean.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '1_val_fe.parquet')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드


# ============================================================
# 0. optuna study파일 db(cat_optuna) 접근, study파일 불러오기
# ============================================================

import optuna

study = optuna.load_study(
    study_name='catboost_1',
    storage=f"sqlite:///{PROJECT_ROOT / 'models' / 'cat_optuna.db'}"
)

print('현재까지 전체 trial 수: ', len(study.trials))
print('현재까지 최적 trial: ', study.best_trial.number)
print('현재까지 최적 RMSE: ', study.best_value)
print('현재까지 최적 hp조합: ', study.best_params)

c:\Users\soohok402\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


현재까지 전체 trial 수:  30
현재까지 최적 trial:  2
현재까지 최적 RMSE:  0.3081064871981181
현재까지 최적 hp조합:  {'depth': 7, 'learning_rate': 0.051059032093947576, 'l2_leaf_reg': 1.0485387725194617, 'random_strength': 8.123245085588687}


In [2]:
# ============================================================
# 1. 라이브러리 불러오기
# ============================================================

import numpy as np
import optuna

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

In [3]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [4]:
# =============================================================
# 2. train 데이터를 튜닝 검증용으로 재분할(val은 최종 val로 남겨둠)
# =============================================================
# 2-1. train -> train_optuna, val_optuna

# 기존 train의 event_id 확인
print(f"train의 event_id 개수: {train['event_id'].nunique()}")  # 98개

optuna_split_idx = int((train['event_id'].nunique()) * 0.8)
print(f"재분할 train_optuna event_id 개수: {optuna_split_idx}")
print()

train_optuna = train.loc[
    train['event_id'] <= optuna_split_idx
]
val_optuna = train.loc[
    train['event_id'] > optuna_split_idx
]

print(f"전체 train event_id 개수: {train['event_id'].nunique()}")
print(f"train_optuna event_id 개수: {train_optuna['event_id'].nunique()}")
print(f"val_optuna event_id 개수: {val_optuna['event_id'].nunique()}")

train의 event_id 개수: 98
재분할 train_optuna event_id 개수: 78

전체 train event_id 개수: 98
train_optuna event_id 개수: 78
val_optuna event_id 개수: 20


In [5]:
# =============================================================
# 3. 종속변수 설정
# =============================================================

y_train_optuna = train_optuna[target_col]
y_val_optuna = val_optuna[target_col]

In [6]:
print(len(train_optuna['station_id'].unique()))
print(len(val_optuna['station_id'].unique()))
print(
    set(train_optuna['station_id'].unique()) - 
    set(val_optuna['station_id'].unique())
    ) # set() 비어있으면 둘은 같은 집합, OHE 문제 없음

465
465
set()


In [6]:
# ============================================================
# 4. station_id 원핫인코딩 독립변수 생성
# ============================================================

# OHE 전 독립변수 데이터셋
X_train_optuna = train_optuna[num_cols_fe]
X_val_optuna = val_optuna[num_cols_fe]

# 원핫인코딩
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train_optuna[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train_optuna[['station_id']])
val_st_id_ohe = ohe.transform(val_optuna[['station_id']])
# 여기까지는 일반적인 인코딩 과정

# train, val 셋에 합치기
from scipy.sparse import hstack
# scipy안에 희소행렬을 다루는 서브캐키지 scipy.sparse
# hstack(horizontal stack)은 concat(axis=1)과 같은 역할, df가 아닌 행렬도 붙일 수 있는 함수

X_train_optuna_ohe = hstack([
    X_train_optuna,
    train_st_id_ohe
]).tocsr()  # df + 희소행렬(ohe) -> csr형태로 저장하여 sparse 상태 유지
# 이 변수는 더이상 df가 아님

X_val_optuna_ohe = hstack([
    X_val_optuna,
    val_st_id_ohe
]).tocsr()

print("원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 늘어남")
print(X_train_optuna.shape, X_train_optuna_ohe.shape)
print(X_val_optuna.shape, X_val_optuna_ohe.shape)
# 제공된 데이터셋의 관측소 개수는 465개(md파일에 명시)
# 기존컬럼 31개 + 465 = 496이 정상

원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 늘어남
(404009, 31) (404009, 496)
(212309, 31) (212309, 496)


In [ ]:
# ============================================================
# 5. optuna objective함수 정의
# ============================================================

# 라이브러리 호출은 위에 해놨음

def objective(trial):

    # 1. 하이퍼파라미터 탐색
    depth = trial.suggest_int(
        'depth',
        4,
        9
    )

    learning_rate = trial.suggest_float(
        'learning_rate',
        0.01,
        0.1,
        log=True
    )

    l2_leaf_reg = trial.suggest_float(
        'l2_leaf_reg',
        1.0,
        10.0,
        log=True
    )

    random_strength = trial.suggest_float(
        'random_strength',
        0.01,
        10.0,
        log=True
    )

    # 2. 모델 정의
    model = CatBoostRegressor(
        iterations=2000,  # 최대 800회 부스팅까지 허용
        depth=depth,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        random_strength=random_strength,
        loss_function='RMSE'
        random_seed=42,  # random_state와 동일한 효과
        thread_count=2,  # RF의 n_jobs와 동일: 병렬작업 사용개수
        verbose=200,  # 100개 트리마다 현황정보 표기
        allow_writing_files=False  # 학습 중 로그 등을 별도로 기록, 분석과 무관하므로 불필요
    )

    # 3. 모델 학습
    model.fit(
        X_train_optuna_ohe,
        y_train_optuna,
        eval_set=(X_val_optuna_ohe, y_val_optuna),  # 내부 검증셋
        early_stopping_rounds=100  # RMSE가 100번 연속 개선되지 않으면 종료
    )

    # 4. 검증 데이터 예측
    pred = model.predict(
        X_val_optuna_ohe
    )

    # 5. RMSE 계산
    rmse = root_mean_squared_error(
        y_val_optuna,
        pred
    )

    # 6. Optuna에 RMSE 반환
    return rmse

In [ ]:
# ============================================================
# 6. Optuna Study 생성 및 실행
# ============================================================

study = optuna.create_study(
    study_name='catboost_1',
    storage=f"sqlite:///{PROJECT_ROOT / 'models' / 'cat_optuna.db'}",
    #sqlite:/// sql db로 다음의 장소에 저장하겠다.
    load_if_exists=True,  # 동일 stuky 존재 시 create X, 불러오기 O
    direction='minimize',
    sampler=optuna.samplers.TPESampler(
        seed=42  # TPESampler의 hp 제안 시 사용하는 난수 고정,
    )            # model의 random_seed와 별도
)

study.optimize(
    objective,
    n_trials=30
)

# 최적 Trial 번호
print('Best Trial:', study.best_trial.number)

# 최저 RMSE
print('Best RMSE:', study.best_value)

# 최적 하이퍼파라미터
print('Best Parameters:', study.best_params)

In [14]:
# Catboost RMSE가 0.2 -> 0.3 이유를 확인하고자 함
# train_optuna, val_optuna, val(기존)
y_val = val[target_col]

for name, y in [
    ('y_train_optuna', y_train_optuna),
    ('y_val_optuna', y_val_optuna),
    ('y_val', y_val)
]:
    print(f'\n{name}')
    print(y.describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)

# 기존에 썼던 기본 CatBoost에 최적파라미터 넣어서 RMSE 평가
from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

model_optuna_cat = CatBoostRegressor(
    random_state = 42)

# 학습
model_optuna_cat.fit(X_train_optuna_ohe, y_train_optuna)

# 예측
pred_optuna_cat = model_optuna_cat.predict(X_val_optuna_ohe)

# 검증
rmse_optuna_cat = root_mean_squared_error(
    y_val_optuna,
    pred_optuna_cat
)

print(rmse_optuna_cat)


y_train_optuna
count    404009.000000
mean          0.099586
std           0.377875
min          -2.180000
50%           0.010000
90%           0.230000
95%           0.490000
99%           1.760000
max           7.980000
Name: target_maxrise_6h, dtype: float64

y_val_optuna
count    212309.000000
mean          0.134761
std           0.445855
min          -2.070000
50%           0.010000
90%           0.380000
95%           0.780000
99%           2.190000
max           8.300000
Name: target_maxrise_6h, dtype: float64

y_val
count    133325.000000
mean          0.096447
std           0.357300
min          -2.120000
50%           0.010000
90%           0.240000
95%           0.510000
99%           1.560000
max           7.710000
Name: target_maxrise_6h, dtype: float64
Learning rate set to 0.105679
0:	learn: 0.3564758	total: 44.9ms	remaining: 44.9s
1:	learn: 0.3383244	total: 82.3ms	remaining: 41.1s
2:	learn: 0.3225444	total: 120ms	remaining: 39.7s
3:	learn: 0.3085940	total: 154ms	remaini

In [10]:
# ============================================================
# 7. 최종검증: 기존 catboost모델, y_val로 검증
# ============================================================

# 종속변수 호출
y_train = train[target_col]
y_val = val[target_col]

# 독립변수 ohe
X_train_fe = train[num_cols_fe]
X_val_fe = val[num_cols_fe]

# 원핫인코딩
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train[['station_id']])
val_st_id_ohe = ohe.transform(val[['station_id']])
# 여기까지는 일반적은 인코딩 과정

# train, val 셋에 합치기
from scipy.sparse import hstack
# scipy안에 희소행렬을 다루는 서브캐키지 scipy.sparse
# hstack(horizontal stack)은 concat(axis=1)과 같은 역할, df가 아닌 행렬도 붙일 수 있는 함수

X_train_with_station_fe = hstack([
    X_train_fe,
    train_st_id_ohe
]).tocsr()  # df + 희소행력(ohe) -> csr형태로 저장하여 sparse 상태 유지
# 이 변수는 더이상 df가 아님

X_val_with_station_fe = hstack([
    X_val_fe,
    val_st_id_ohe
]).tocsr()

print("원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남")
print(X_train_fe.shape, X_train_with_station_fe.shape)
print(X_val_fe.shape, X_val_with_station_fe.shape)
# 제공된 데이터셋의 관측소 개수는 465개(md파일에 명시)
# 기존컬럼 31개 + 465 = 496이 정상

원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남
(616318, 31) (616318, 496)
(133325, 31) (133325, 496)


In [ ]:
# ============================================================
# 7. 모델15 (최종검증: 기존 catboost모델, y_val)
# ============================================================

print(study.best_params)

model = CatBoostRegressor(
    **study.best_params,  # study 폴더에서 최적 hp 불러오기    
    random_state=42
)

model.fit(X_train_with_station_fe, y_train)

pred_val_optuna_tuned = model.predict(X_val_with_station_fe)
pred_train_optuna_tuned = model.predict(X_train_with_station_fe)

rmse_val_optuna_tuned = root_mean_squared_error(
    y_val,
    pred_val_optuna_tuned
)

rmse_train_optuna_tuned = root_mean_squared_error(
    y_train,
    pred_train_optuna_tuned
)

print('val RMSE: ', rmse_val_optuna_tuned)
print('train RMSE: ', rmse_train_optuna_tuned)
print(
    'Gap: ',
    rmse_val_optuna_tuned - rmse_train_optuna_tuned
)


In [9]:
# ============================================================
# 7. 모델14 (튜닝용 내부검증셋으로 돌려보기, 결과확인용)
# ============================================================

# 모델 정의
model_optuna_split = CatBoostRegressor(
    **study.best_params,
    iterations=2000,
    loss_function='RMSE',
    random_seed=42,
    thread_count=2,
    verbose=200,
    allow_writing_files=False
)

# 학습 (Optuna 당시 조건 재현)
model_optuna_split.fit(
    X_train_optuna_ohe,
    y_train_optuna,
    eval_set=(X_val_optuna_ohe, y_val_optuna),
    early_stopping_rounds=100
)

pred_val_optuna_tuned_split = model_optuna_split.predict(X_val_optuna_ohe)
pred_train_optuna_tuned_split = model_optuna_split.predict(X_train_optuna_ohe)

rmse_val_optuna_tuned_split = root_mean_squared_error(
    y_val_optuna,
    pred_val_optuna_tuned_split
)

rmse_train_optuna_tuned_split = root_mean_squared_error(
    y_train_optuna,
    pred_train_optuna_tuned_split
)

print('val RMSE: ', rmse_val_optuna_tuned_split)
print('train RMSE: ', rmse_train_optuna_tuned_split)
print(
    'Gap: ',
    rmse_val_optuna_tuned_split - rmse_train_optuna_tuned_split
)


0:	learn: 0.3681010	test: 0.4394793	best: 0.4394793 (0)	total: 65.5ms	remaining: 2m 10s
200:	learn: 0.1910971	test: 0.3174701	best: 0.3174701 (200)	total: 12.9s	remaining: 1m 55s
400:	learn: 0.1762266	test: 0.3124704	best: 0.3124704 (400)	total: 25.2s	remaining: 1m 40s
600:	learn: 0.1666130	test: 0.3104160	best: 0.3104160 (600)	total: 37.2s	remaining: 1m 26s
800:	learn: 0.1596830	test: 0.3094214	best: 0.3094163 (799)	total: 49.1s	remaining: 1m 13s
1000:	learn: 0.1543291	test: 0.3086345	best: 0.3086135 (993)	total: 1m 1s	remaining: 1m
1200:	learn: 0.1498188	test: 0.3083157	best: 0.3082572 (1175)	total: 1m 13s	remaining: 49.1s
1400:	learn: 0.1458675	test: 0.3081555	best: 0.3081065 (1361)	total: 1m 26s	remaining: 36.9s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.308106487
bestIteration = 1361

Shrink model to first 1362 iterations.
val RMSE:  0.3081064871981181
train RMSE:  0.1466308116315946
Gap:  0.1614756755665235
